In [1]:
print("OK!")

OK!


In [18]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.llms import CTransformers 
import os
from dotenv import load_dotenv


In [3]:
load_dotenv()

True

In [4]:
PINECONE_API_KEY =os.getenv("PINECONE_API_KEY")

In [5]:
if not PINECONE_API_KEY:
    raise ValueError("PINECONE_API_KEY required")

In [6]:
# Extract data from the PDF

def load_pdf(data):
    loader = DirectoryLoader(data,
                    glob="*.pdf",
                    loader_cls=PyPDFLoader)

    documents = loader.load()
    
    return documents

In [7]:
extracted_data = load_pdf("data/")

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Type': '/Font', '/Encoding': '/WinAnsiEncoding', '/BaseFont': '/PKNADH+Berkeley-BookItalic', '/FirstChar': 32, '/LastChar': 111, '/Subtype': '/Type1', '/FontDescriptor': IndirectObject(8788, 0, 2765039944000), '/Widths': [241, 241, 352, 481, 481, 667, 778, 204, 315, 315, 500, 481, 241, 333, 241, 278, 481, 481, 481, 481, 481, 481, 481, 481, 481, 481, 241, 241, 481, 481, 481, 352, 800, 593, 519, 611, 685, 537, 519, 667, 685, 296, 296, 611, 500, 796, 685, 685, 519, 667, 556, 463, 556, 685, 593, 870, 593, 519, 556, 315, 278, 315, 481, 500, 241, 463, 444, 370, 463, 389, 259, 407, 463, 241, 241, 444, 241, 722, 481, 426]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Type': '/Font', '/Encoding': '/WinAnsiEncoding', '/BaseFont': '/PKNAFK+Berkeley-Bold', '/FirstChar': 32,

In [8]:
# extracted_data

In [9]:
# Create text chunks
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap=20)
    text_chunks = text_splitter.split_documents(extracted_data)

    return text_chunks

In [10]:
text_chunks = text_split(extracted_data)
print("length of my chunk:", len(text_chunks))

length of my chunk: 5861


In [11]:
# text_chunks

In [12]:
# download the embedding model
def download_hugging_face_embeddings():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

In [ ]:
embeddings = download_hugging_face_embeddings()

In [14]:
embeddings

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [15]:
query_result = embeddings.embed_query("Hello world")
print("Length", len(query_result))

Length 384


In [17]:
# query_result

In [ ]:
# Initialize pinecone

pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "medical-chatbot"
# Created index if index does not exists
if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension= 384,
        metric ="cosine",
        spec=ServerlessSpec(cloud="aws",region="us-east-1")
    )

In [27]:
index_count = pc.Index(index_name).describe_index_stats().total_vector_count

# Creating Embeddings for Each of the Text Chunks & storing
if index_count != len(text_chunks):
    doc_search = PineconeVectorStore.from_texts(
    [t.page_content for t in text_chunks], embeddings, index_name=index_name
    )
else:
    doc_search = PineconeVectorStore(index_name=index_name, embedding=embeddings)

In [28]:

docsearch = PineconeVectorStore.from_existing_index(index_name,embeddings)

query = "What is Acupuncture"

docs = docsearch.similarity_search(query, k=3)

print("Result", docs)

Result [Document(id='50ae741f-b463-4662-9221-7a4ae8b0f34d', metadata={}, page_content='for headaches and chronic pain, associated with problems\nlike back injuries and arthritis. It has also been used to\nsupplement invasive Western treatments like chemother-\napy and surgery. Acupuncture is generally most effective\nwhen used as prevention or before a health condition\nbecomes acute, but it has been used to help patients suf-\nfering from cancer and AIDS. Acupuncture is limited in\ntreating conditions or traumas that require surgery or\nemergency care (such as for broken bones).'), Document(id='5cd7d240-19ae-4ec0-9732-253d43ffe49a', metadata={}, page_content='instead of with the insertion of needles. Acupressure is\nused to relieve a variety of symptoms and pain.\nPurpose\nAcupressure massage performed by a therapist can be\nvery effective both as prevention and as a treatment for many\nhealth conditions, including headaches, general aches and\npains, colds and flu, arthritis,allergie

In [29]:
prompt_template="""
Use the following pieces of information to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context: {context}
Question: {question}

Only return the helpful answer below and nothing else.
Helpful answer:
"""

In [30]:
PROMPT = PromptTemplate(template=prompt_template,input_variables=["context","question"])
chain_type_kwargs={"prompt": PROMPT}

In [31]:
llm=CTransformers(model="model/llama-2-7b-chat.ggmlv3.q4_0.bin",
                model_type="llama",
                config={'max_new_tokens': 512,
                'temperature': 0.8})

In [32]:
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=docsearch.as_retriever(search_kwargs={'k':2}),
    return_source_documents=True,
    chain_type_kwargs=chain_type_kwargs
)

In [ ]:
while True:
    user_input = input(f"Input Prompt:")
    result =qa.invoke({"query": user_input})
    print("Response : ", result["result"])

Number of tokens (513) exceeded maximum context length (512).
Number of tokens (514) exceeded maximum context length (512).
Number of tokens (515) exceeded maximum context length (512).
Number of tokens (516) exceeded maximum context length (512).
Number of tokens (517) exceeded maximum context length (512).
Number of tokens (518) exceeded maximum context length (512).
Number of tokens (519) exceeded maximum context length (512).
Number of tokens (520) exceeded maximum context length (512).
Number of tokens (521) exceeded maximum context length (512).
Number of tokens (522) exceeded maximum context length (512).
Number of tokens (523) exceeded maximum context length (512).
Number of tokens (524) exceeded maximum context length (512).
Number of tokens (525) exceeded maximum context length (512).
Number of tokens (526) exceeded maximum context length (512).
Number of tokens (527) exceeded maximum context length (512).
Number of tokens (528) exceeded maximum context length (512).
Number o

Response :  Albinism is a rare genetic disorder that affects the production of melanin, which gives color to the skin, hair, and eyes. It can lead to vision problems, including nystagmus (involuntary eye movements), photophobia (sensitivity to light), and visual impairment. In addition, people with albinism are at a higher risk of developing certain health problems, such as bleeding disorders, inflammation of the large bowel, lung disease, and kidney problems. There are several types of albinism, including oculocutaneous albinism, which affects the eyes, hair, and skin. Diagnosis can be challenging, but recent advances in testing have made it possible to identify some easier to identify carriers more possible to identify carriers possible to diagnosible to identified carries to identify carriers possible to identifyable to identify some what is possible to identify carriers possible to identify some what we can identify carriers possible to identify some easier to identify carriers pos